### 01. Environment Setup

The SOCOFing dataset will be used as the source dataset. We will later select only two subjects and map their original IDs to `User_01` and `User_02`.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import kagglehub
import shutil

# The original project path was hard-coded to one computer. Run this notebook from the project root instead.
PROJECT_PATH = Path.cwd().resolve()

DATASET_PATH = PROJECT_PATH / "Dataset" / "Raw" / "Fingerprint"

OUTPUT_PATH = PROJECT_PATH / "Dataset" / "Processed" / "Fingerprint"

SELECTED_PATH = OUTPUT_PATH / "Selected_Images"

NOTEBOOK_PATH = PROJECT_PATH / "Notebooks" / "Fingerprint"

MODEL_PATH = PROJECT_PATH / "Models" / "Fingerprint"

USERS = ["User_01", "User_02"]

print("Fingerprint Environment Configured")
print("---------------------------------------------")
print(f"Project Path: {PROJECT_PATH}")
print(f"Raw Dataset Path: {DATASET_PATH}")
print(f"Processed Path: {OUTPUT_PATH}")
print(f"Notebook Path: {NOTEBOOK_PATH}")
print(f"Model Path: {MODEL_PATH}")


### 02. Download SOCOFing Dataset

In [ ]:

# Download one fingerprint image as a test

import kagglehub
import shutil

DATASET_HANDLE = "ruizgara/socofing"

TEST_FILE = "SOCOFing/Real/100__M_Left_index_finger.BMP"

TEST_OUTPUT_PATH = DATASET_PATH / "Download_Test"
TEST_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

downloaded_file = Path(
    kagglehub.dataset_download(
        DATASET_HANDLE,
        path=TEST_FILE
    )
)

test_image_path = TEST_OUTPUT_PATH / "User_01_Left_index_finger.BMP"

shutil.copy2(downloaded_file, test_image_path)

print("Fingerprint Test Download Completed")
print("---------------------------------------------")
print(f"Downloaded File: {downloaded_file}")
print(f"Saved File: {test_image_path}")
print(f"File Exists: {test_image_path.exists()}")
print(f"File Size: {test_image_path.stat().st_size / 1024:.2f} KB")


## 03. Download Selected Subjects

In this section, we download the original fingerprint images of two selected subjects from the SOCOFing Real directory.

The original subject IDs are mapped to the project's user labels:

Subject 100 → User_01

Subject 101 → User_02

Each subject has ten fingerprint images representing five fingers from each hand.

In [ ]:

# Configure Selected Subjects

SUBJECTS = {
    "100": "User_01",
    "101": "User_02"
}

FINGERPRINTS = [
    "Left_index_finger",
    "Left_little_finger",
    "Left_middle_finger",
    "Left_ring_finger",
    "Left_thumb_finger",
    "Right_index_finger",
    "Right_little_finger",
    "Right_middle_finger",
    "Right_ring_finger",
    "Right_thumb_finger"
]

# Download Selected Fingerprint Images

for subject_id, user_id in SUBJECTS.items():

    user_path = DATASET_PATH / user_id
    user_path.mkdir(parents=True, exist_ok=True)

    for fingerprint in FINGERPRINTS:

        filename = f"{subject_id}__M_{fingerprint}.BMP"
        relative_path = f"SOCOFing/Real/{filename}"

        output_file = user_path / f"{user_id}_{fingerprint}.BMP"

        # Reuse the previously downloaded test image if available
        if subject_id == "100" and fingerprint == "Left_index_finger":
            shutil.copy2(test_image_path, output_file)

        else:
            downloaded_file = Path(
                kagglehub.dataset_download(
                    DATASET_HANDLE,
                    path=relative_path
                )
            )

            shutil.copy2(downloaded_file, output_file)

        print(f"Downloaded: {user_id} - {fingerprint}")

print("\nSelected Fingerprint Images Downloaded")


### 04. Dataset Verification

In [ ]:

# Verify Selected Fingerprint Dataset

for user_id in SUBJECTS.values():

    user_path = DATASET_PATH / user_id
    image_files = list(user_path.glob("*.BMP"))

    print(f"{user_id}: {len(image_files)} images found.")

    for image_file in sorted(image_files):
        print(f"  {image_file.name}")

total_images = sum(
    len(list((DATASET_PATH / user_id).glob("*.BMP")))
    for user_id in SUBJECTS.values()
)

print("\nDataset Summary")
print("---------------------------------------------")
print(f"Total Subjects: {len(SUBJECTS)}")
print(f"Total Fingerprint Images: {total_images}")
print(f"Expected Images: 20")
print(f"Dataset Validation: {'Passed' if total_images == 20 else 'Check Required'}")


### 05. Fingerprint Image Inspection

In this section, we inspect the original fingerprint images before preprocessing.

We examine image dimensions, data types, pixel intensity ranges, and visual quality to understand the dataset characteristics and identify any unreadable or corrupted images.

In [ ]:

# Inspect Fingerprint Images

fingerprint_records = []
fingerprint_images = {}

for user_id in USERS:

    user_path = DATASET_PATH / user_id

    for image_path in sorted(user_path.glob("*.BMP")):

        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

        if image is None:
            print(f"Warning: Unable to read {image_path.name}")
            continue

        fingerprint_images[(user_id, image_path.name)] = image

        fingerprint_records.append({
            "User_ID": user_id,
            "Image_Name": image_path.name,
            "Height": image.shape[0],
            "Width": image.shape[1],
            "Data_Type": str(image.dtype),
            "Min_Pixel_Value": int(image.min()),
            "Max_Pixel_Value": int(image.max())
        })

fingerprint_metadata_df = pd.DataFrame(fingerprint_records)

print("Fingerprint Image Inspection")
print("---------------------------------------------")
print(f"Readable Images: {len(fingerprint_metadata_df)}")
print(f"Unreadable Images: {20 - len(fingerprint_metadata_df)}")

display(fingerprint_metadata_df)

print("\nImage Dimensions Distribution:")
display(
    fingerprint_metadata_df.groupby(
        ["Height", "Width"]
    ).size().reset_index(name="Count")
)


### 06. Visual Fingerprint Inspection

In [ ]:

# Visualize Fingerprints and Generate Inspection Summary

visual_records = []

fig, axes = plt.subplots(2, 10, figsize=(22, 7))

for row, user_id in enumerate(USERS):

    for col, finger in enumerate(finger_order):

        image_name = f"{user_id}_{finger}.BMP"
        image = fingerprint_images[(user_id, image_name)]

        # Display original fingerprint
        axes[row, col].imshow(image, cmap="gray")
        axes[row, col].set_title(finger.replace("_", "\n"), fontsize=8)
        axes[row, col].axis("off")

        # Calculate image statistics
        visual_records.append({
            "User_ID": user_id,
            "Image_Name": image_name,
            "Mean_Intensity": round(float(np.mean(image)), 2),
            "Std_Intensity": round(float(np.std(image)), 2),
            "Dark_Pixel_Ratio_%": round(float(np.mean(image < 50) * 100), 2),
            "Bright_Pixel_Ratio_%": round(float(np.mean(image > 200) * 100), 2)
        })

    axes[row, 0].set_ylabel(user_id, fontsize=12, labelpad=15)

plt.suptitle("Original Fingerprint Images - User Comparison", fontsize=15)
plt.tight_layout()
plt.show()

# Generate numerical summary
visual_inspection_df = pd.DataFrame(visual_records)

print("\nFINGERPRINT VISUAL INSPECTION")
print("=" * 60)

print("\n1. Overall Statistics")
display(
    visual_inspection_df[
        ["Mean_Intensity", "Std_Intensity",
         "Dark_Pixel_Ratio_%", "Bright_Pixel_Ratio_%"]
    ].describe().round(2)
)

print("\n2. Statistics by User")
display(
    visual_inspection_df.groupby("User_ID")[
        ["Mean_Intensity", "Std_Intensity",
         "Dark_Pixel_Ratio_%", "Bright_Pixel_Ratio_%"]
    ].mean().round(2)
)

print("\n3. Individual Image Statistics")
display(visual_inspection_df)

# Copy-friendly text output
print("\n4. COPY-READY SUMMARY")
print(visual_inspection_df.to_string(index=False))
